In [ ]:
import pandas as pd
import numpy as np

In [ ]:
model_data = pd.read_csv("../data/processed/model_data.csv")
ranking = pd.read_csv("../data/raw/fifa_ranking.csv")

model_data["date"] = pd.to_datetime(model_data["date"])

In [ ]:
# Clean column names properly
ranking.columns = (
    ranking.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(".", "_")
)

ranking.columns

In [ ]:
ranking = ranking.rename(columns={
    "total_points": "team_rank_points"
})

ranking["rank_date"] = pd.to_datetime(ranking["rank_date"])

ranking = ranking[[
    "rank_date",
    "team",
    "team_rank",
    "team_rank_points"
]]

ranking.head()

In [1]:
import pandas as pd
import numpy as np

# Load files again
model_data = pd.read_csv("../data/processed/model_data.csv")
ranking = pd.read_csv("../data/raw/fifa_ranking.csv")

# Convert match date
model_data["date"] = pd.to_datetime(model_data["date"])

# Clean ranking column names
ranking.columns = (
    ranking.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(".", "_", regex=False)
)

# Show cleaned columns
print(ranking.columns)

# Rename points column
ranking = ranking.rename(columns={
    "total_points": "team_rank_points"
})

# Some files have both date and rank_date.
# We will safely use rank_date if it exists, otherwise use date.
if "rank_date" in ranking.columns:
    ranking["rank_date"] = pd.to_datetime(ranking["rank_date"])
elif "date" in ranking.columns:
    ranking["rank_date"] = pd.to_datetime(ranking["date"])
else:
    raise ValueError("No ranking date column found.")

# Keep only the needed columns
ranking = ranking[[
    "rank_date",
    "team",
    "team_rank",
    "team_rank_points"
]]

ranking.head()

Index(['date', 'semester', 'rank', 'team', 'acronym', 'total_points',
       'previous_points', 'diff_points'],
      dtype='object')


KeyError: "['team_rank'] not in index"

In [1]:
import pandas as pd
import numpy as np

# Load files
model_data = pd.read_csv("../data/processed/model_data.csv")
ranking = pd.read_csv("../data/raw/fifa_ranking.csv")

# Convert model_data date
model_data["date"] = pd.to_datetime(model_data["date"])

# Clean ranking column names
ranking.columns = (
    ranking.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(".", "_", regex=False)
)

print("Cleaned ranking columns:")
print(ranking.columns.tolist())

# Rename columns safely based on what exists
rename_map = {}

if "rank" in ranking.columns:
    rename_map["rank"] = "team_rank"

if "team_rank" in ranking.columns:
    rename_map["team_rank"] = "team_rank"

if "country_full" in ranking.columns:
    rename_map["country_full"] = "team"

if "team" in ranking.columns:
    rename_map["team"] = "team"

if "total_points" in ranking.columns:
    rename_map["total_points"] = "team_rank_points"

if "previous_points" in ranking.columns and "total_points" not in ranking.columns:
    rename_map["previous_points"] = "team_rank_points"

ranking = ranking.rename(columns=rename_map)

# Create rank_date safely
if "rank_date" in ranking.columns:
    ranking["rank_date"] = pd.to_datetime(ranking["rank_date"])
elif "date" in ranking.columns:
    ranking["rank_date"] = pd.to_datetime(ranking["date"])
else:
    raise ValueError("No date column found in ranking file.")

print("Columns after rename:")
print(ranking.columns.tolist())

# Check required columns
required_cols = ["rank_date", "team", "team_rank", "team_rank_points"]

missing_cols = [col for col in required_cols if col not in ranking.columns]

if missing_cols:
    print("Missing columns:", missing_cols)
    print("Available columns:", ranking.columns.tolist())
else:
    ranking = ranking[required_cols]
    print("Ranking table ready.")
    display(ranking.head())

Cleaned ranking columns:
['date', 'semester', 'rank', 'team', 'acronym', 'total_points', 'previous_points', 'diff_points']
Columns after rename:
['date', 'semester', 'team_rank', 'team', 'acronym', 'team_rank_points', 'previous_points', 'diff_points', 'rank_date']
Ranking table ready.


,rank_date,team,team_rank,team_rank_points
0,1970-01-01 00:00:00.000002024,Argentina,1,1867.25
1,1970-01-01 00:00:00.000002024,France,2,1859.78
2,1970-01-01 00:00:00.000002024,Spain,3,1853.27
3,1970-01-01 00:00:00.000002024,England,4,1813.81
4,1970-01-01 00:00:00.000002024,Brazil,5,1775.85


In [2]:
model_data = model_data.sort_values("date").reset_index(drop=True)
ranking = ranking.sort_values("rank_date").reset_index(drop=True)

In [3]:
home_ranking = ranking.rename(columns={
    "team": "home_team",
    "team_rank": "home_team_rank",
    "team_rank_points": "home_team_rank_points",
    "rank_date": "home_rank_date"
})

model_data = pd.merge_asof(
    model_data.sort_values("date"),
    home_ranking.sort_values("home_rank_date"),
    left_on="date",
    right_on="home_rank_date",
    by="home_team",
    direction="backward"
)

model_data[[
    "date",
    "home_team",
    "home_team_rank",
    "home_team_rank_points"
]].head()

,date,home_team,home_team_rank,home_team_rank_points
0,1872-11-30,Scotland,NaN,NaN
1,1873-03-08,England,NaN,NaN
2,1874-03-07,Scotland,NaN,NaN
3,1875-03-06,England,NaN,NaN
4,1876-03-04,Scotland,NaN,NaN


In [4]:
away_ranking = ranking.rename(columns={
    "team": "away_team",
    "team_rank": "away_team_rank",
    "team_rank_points": "away_team_rank_points",
    "rank_date": "away_rank_date"
})

model_data = pd.merge_asof(
    model_data.sort_values("date"),
    away_ranking.sort_values("away_rank_date"),
    left_on="date",
    right_on="away_rank_date",
    by="away_team",
    direction="backward"
)

model_data[[
    "date",
    "home_team",
    "away_team",
    "home_team_rank",
    "away_team_rank",
    "home_team_rank_points",
    "away_team_rank_points"
]].head()

,date,home_team,away_team,home_team_rank,away_team_rank,home_team_rank_points,away_team_rank_points
0,1872-11-30,Scotland,England,NaN,NaN,NaN,NaN
1,1873-03-08,England,Scotland,NaN,NaN,NaN,NaN
2,1874-03-07,Scotland,England,NaN,NaN,NaN,NaN
3,1875-03-06,England,Scotland,NaN,NaN,NaN,NaN
4,1876-03-04,Scotland,England,NaN,NaN,NaN,NaN


In [5]:
model_data["rank_difference"] = (
    model_data["away_team_rank"] - model_data["home_team_rank"]
)

model_data["rank_points_difference"] = (
    model_data["home_team_rank_points"] - model_data["away_team_rank_points"]
)

In [6]:
model_data[[
    "home_team_rank",
    "away_team_rank",
    "home_team_rank_points",
    "away_team_rank_points",
    "rank_difference",
    "rank_points_difference"
]].isnull().sum()

home_team_rank            12774
away_team_rank            12652
home_team_rank_points     12774
away_team_rank_points     12652
rank_difference           15887
rank_points_difference    15887
dtype: int64

In [7]:
model_data_ranked = model_data.dropna(subset=[
    "home_team_rank",
    "away_team_rank",
    "home_team_rank_points",
    "away_team_rank_points"
]).copy()

model_data.shape, model_data_ranked.shape

((49819, 28), (33932, 28))

In [8]:
model_data_ranked.to_csv("../data/processed/model_data_with_rankings.csv", index=False)